# 01 · Perfilado de la zona Landing
Evidencia de lectura y exploración de datos que pide la consigna.Este notebook **solo lee** de `landing/`: no escribe en el Data Lake y no modifica nada, en líneacon. En esta etapa el objetivo es conocer el dato, no construir el pipeline.Cada resultado que sale de acá está citado en `decisions.md`.

In [ ]:
import json, glob, os, collections, statistics
from pathlib import Path
import pandas as pd

# Busca el dataset en el repo.
# En Colab: from google.colab import drive; drive.mount('/content/drive')
#           CANDIDATOS = [Path('/content/drive/MyDrive/bigdata_itba/datalake/landing')]
CANDIDATOS = [
    Path('../data/datalake/landing'),
]
LANDING = next((p for p in CANDIDATOS if p.exists()), None)
assert LANDING, "No encuentro landing. Copiá el dataset a data/datalake/landing/."

CSVS = sorted(p.name for p in LANDING.glob('*.csv'))
PARTS = sorted(LANDING.glob('usage_events_stream/*.jsonl'))

print(f"Landing: {LANDING.resolve()}")
print(f"CSV: {len(CSVS)} | partes JSONL: {len(PARTS)}")

REPORT = []
def log(s=""):
    print(s)


### 1 · Inventario: filas, tamaño, clave primaria y duplicados

In [ ]:
PK = {
    'customers_orgs.csv':   ['org_id'],
    'users.csv':            ['user_id'],
    'resources.csv':        ['resource_id'],
    'support_tickets.csv':  ['ticket_id'],
    'marketing_touches.csv':['touch_id'],
    'nps_surveys.csv':      ['org_id','survey_date'],
    'billing_monthly.csv':  ['invoice_id'],
}

dfs, inv = {}, []
for name in CSVS:
    df = pd.read_csv(LANDING/name, dtype=str, keep_default_na=False, na_values=[''])
    dfs[name] = df
    k = PK[name]
    inv.append({'fuente':name, 'filas':len(df), 'columnas':df.shape[1],
                'PK':'+'.join(k), 'duplicados_PK':int(df.duplicated(subset=k).sum()),
                'KB':round((LANDING/name).stat().st_size/1024,1)})

n_events = sum(sum(1 for _ in open(p)) for p in PARTS)
mb = sum(p.stat().st_size for p in PARTS)/1024/1024
inv.append({'fuente':'usage_events_stream/*.jsonl','filas':n_events,'columnas':13,
            'PK':'event_id','duplicados_PK':None,'KB':round(mb*1024,1)})

inventario = pd.DataFrame(inv)
log("### Inventario\n")
log(inventario.to_markdown(index=False))
log(f"\nTotal: {inventario.filas.sum():,} filas · {inventario.KB.sum()/1024:.1f} MB")
inventario

### 2 · Nulos por columna

In [ ]:
log("\n### Nulos por columna\n")
for name, df in dfs.items():
    nulos = {c:int(df[c].isna().sum()) for c in df.columns if df[c].isna().any()}
    if nulos:
        detalle = ", ".join(f"`{c}` {v} ({100*v/len(df):.1f}%)" for c,v in nulos.items())
        log(f"- **{name}** — {detalle}")
    else:
        log(f"- **{name}** — sin nulos")

### 3 · Dominios categóricos
Se verifica si hay variantes que requieran conformance (`us-east` vs `US_East`, etc.).

In [ ]:
CATS = {
    'customers_orgs.csv': ['industry','hq_region','plan_tier','lifecycle_stage','marketing_source'],
    'users.csv':          ['role','active'],
    'resources.csv':      ['service','region','state'],
    'support_tickets.csv':['category','severity','sla_breached'],
    'marketing_touches.csv':['channel','converted'],
    'billing_monthly.csv':['month','currency'],
}
log("\n### Dominios categóricos\n")
for name, cols in CATS.items():
    for c in cols:
        vc = dfs[name][c].value_counts()
        log(f"- `{name}::{c}` → {len(vc)} valores: " + ", ".join(f"{k} ({v})" for k,v in vc.items()))
log("\n**Sin variantes de formato**: ninguna región, servicio o plan aparece con mayúsculas, "
    "guiones bajos o espacios distintos. La función de conformance se implementa igual, "
    "porque una v2 del dataset podría no venir limpia.")

### 4 · Problemas de calidad medidos

In [ ]:
log("\n### Problemas de calidad\n")
hall = []

b = dfs['billing_monthly.csv'].copy()
b['fx'] = b.exchange_rate_to_usd.astype(float)
b['sub_num'] = b.subtotal.astype(float)
usd = b[b.currency=='USD']
usd_bad = usd[(usd.fx-1).abs() > 0.001]
hall.append(("billing · currency=USD con fx != 1", f"{len(usd_bad)}/{len(usd)}",
             f"rango {usd.fx.min():.4f}–{usd.fx.max():.4f} → distorsiona el revenue hasta ±15% (regla R5)"))
hall.append(("billing · subtotal negativo", f"{int((b.sub_num<0).sum())}/{len(b)}",
             f"mínimo {b.sub_num.min():.2f} → notas de crédito, NO se bloquean"))
hall.append(("billing · credits nulo", f"{int(b.credits.isna().sum())}/{len(b)}",
             "se interpreta como 0, no como faltante (supuesto 3)"))
for cur in ['USD','EUR','ARS']:
    s = b[b.currency==cur].fx
    hall.append((f"billing · fx {cur}", f"{len(s)} filas",
                 f"min {s.min():.4f} / mediana {s.median():.4f} / max {s.max():.4f}"))

t = dfs['support_tickets.csv'].copy()
csat = pd.to_numeric(t.csat, errors='coerce')
fuera = csat.notna() & ~csat.between(1,5)
hall.append(("tickets · csat fuera de [1,5]", f"{int(fuera.sum())}/{int(csat.notna().sum())}",
             "valores " + str(sorted(csat[fuera].unique().tolist())) + " (regla R6)"))
hall.append(("tickets · resolved_at nulo", f"{int(t.resolved_at.isna().sum())}/{len(t)}",
             "tickets abiertos, NO es un defecto"))
mal = t.dropna(subset=['resolved_at'])
hall.append(("tickets · resolved_at < created_at", f"{int((mal.resolved_at<mal.created_at).sum())}", "ninguno"))
hall.append(("tickets · SLA breach", f"{int((t.sla_breached=='True').sum())}/{len(t)}",
             "por severidad: " + str(t[t.sla_breached=='True'].severity.value_counts().to_dict())))

o = pd.to_numeric(dfs['customers_orgs.csv'].nps_score, errors='coerce')
hall.append(("orgs · nps_score fuera de [-100,100]", f"{int((o.notna() & ~o.between(-100,100)).sum())}",
             f"valor {o[o>100].tolist()} (regla R7)"))

calidad = pd.DataFrame(hall, columns=['problema','magnitud','interpretación'])
log(calidad.to_markdown(index=False))
calidad

### 5 · Integridad referencial

In [ ]:
log("\n### Integridad referencial\n")
orgs = set(dfs['customers_orgs.csv'].org_id)
for name in ['users.csv','resources.csv','support_tickets.csv','marketing_touches.csv',
             'nps_surveys.csv','billing_monthly.csv']:
    s = set(dfs[name].org_id)
    log(f"- `{name}` → {len(s)} orgs distintas, **{len(s-orgs)} huérfanas**")
log("\nTodas las fuentes apuntan a las 80 organizaciones de `customers_orgs.csv`. Sin huérfanos.")

### 6 · Eventos: esquema, evolución v1→v2 y tipos ambiguos

In [ ]:
SAMPLE = PARTS
keys=collections.Counter(); sv=collections.Counter(); svc=collections.Counter()
met=collections.Counter(); unit_by_metric=collections.defaultdict(collections.Counter)
vtypes=collections.Counter(); ids=collections.Counter()
costs=[]; dates=collections.Counter(); v1d=[]; v2d=[]
unit_null_val_ok=0; genai=collections.Counter(); n=0
for p in SAMPLE:
    for line in open(p):
        d=json.loads(line); n+=1
        ids[d['event_id']]+=1; keys.update(d.keys())
        sv[d['schema_version']]+=1; svc[d['service']]+=1; met[d['metric']]+=1
        unit_by_metric[d['metric']][d.get('unit')]+=1
        v=d.get('value'); vtypes[type(v).__name__]+=1
        if d.get('unit') is None and v is not None: unit_null_val_ok+=1
        costs.append(d['cost_usd_increment'])
        day=d['timestamp'][:10]; dates[day]+=1
        (v2d if d['schema_version']==2 else v1d).append(day)
        if 'genai_tokens' in d: genai[d['service']]+=1

log(f"\n### Eventos · {n:,} filas en {len(SAMPLE)} archivos\n")
log(f"- **event_id únicos:** {len(ids):,} de {n:,} → **{n-len(ids)} duplicados**")
log(f"- **Rango temporal:** {min(dates)} → {max(dates)} ({len(dates)} días, ~{n//len(dates)} eventos/día)")
log(f"- **schema_version:** v1 {sv[1]:,} ({100*sv[1]/n:.0f}%) · v2 {sv[2]:,} ({100*sv[2]/n:.0f}%)")
log(f"- **Corte de esquema:** v1 cubre {min(v1d)} → {max(v1d)} · v2 cubre {min(v2d)} → {max(v2d)} — sin solapamiento")
log(f"- **`genai_tokens`** aparece solo en `service=genai` y `schema_version=2`: {sum(genai.values()):,} de {svc['genai']:,} eventos genai")
log(f"- **Servicios:** {dict(svc)}")
log(f"- **Métricas:** {dict(met)} — las 3 aparecen en los 6 servicios")
for m,u in unit_by_metric.items():
    log(f"  - `{m}` → {dict(u)}")
log(f"\n**Tipos de `value`: {dict(vtypes)}**")
log(f"→ {vtypes['str']} strings ({100*vtypes['str']/n:.2f}%). Declarar `DoubleType` los convierte "
    f"en null **sin error**: los nulos pasarían de {100*vtypes['NoneType']/n:.2f}% a "
    f"{100*(vtypes['NoneType']+vtypes['str'])/n:.2f}%. Por eso D10 lee `value` como string en Bronze.")
log(f"\n**`unit` nulo con `value` presente:** {unit_null_val_ok} ({100*unit_null_val_ok/n:.2f}%) → "
    "imputable por `metric`, que es 1:1 con `unit` (regla R3).")

In [ ]:
costs.sort()
q = lambda p: costs[int(p*len(costs))]
log("\n### Distribución de `cost_usd_increment`\n")
dist = pd.DataFrame([{'min':costs[0],'p50':q(.5),'media':sum(costs)/len(costs),'p95':q(.95),
                      'p99':q(.99),'p99.9':q(.999),'max':costs[-1]}]).round(2)
log(dist.to_markdown(index=False))
neg = sum(1 for c in costs if c < -0.01)
log(f"\n- **Costos negativos:** {neg} ({100*neg/len(costs):.2f}%) → regla R2, se marcan como crédito")
log(f"- **Spikes > 100 USD:** {sum(1 for c in costs if c>100)}")
log(f"- La **media ({sum(costs)/len(costs):.2f}) es {sum(costs)/len(costs)/q(.5):.1f}× la mediana ({q(.5):.2f})** "
    "y el salto de p99 a p99.9 es de 6×: distribución muy asimétrica. "
    "Por eso D20 usa MAD y no z-score: media y desvío están contaminados por los mismos outliers que se busca detectar.")
dist

### 7 · La trampa del watermark

In [ ]:
import datetime as dt

def simular(wm_days, batch_size=5, files=None):
    files = files or PARTS
    maxseen=None; dropped=0; total=0
    for i in range(0, len(files), batch_size):
        evs=[dt.datetime.strptime(json.loads(l)['timestamp'], '%Y-%m-%dT%H:%M:%SZ')
             for f in files[i:i+batch_size] for l in open(f)]
        wm = maxseen - dt.timedelta(days=wm_days) if maxseen else None
        for t in evs:
            total += 1
            if wm and t < wm: dropped += 1
        m = max(evs)
        maxseen = m if maxseen is None or m > maxseen else maxseen
    return total, dropped

log("\n### Impacto del watermark\n")
rows=[]
for wm in (1,2,7,30,60):
    tot,dr = simular(wm)
    rows.append({'watermark':f'{wm} días','eventos':tot,'descartados':dr,'% descartado':round(100*dr/tot,1)})
wmdf = pd.DataFrame(rows)
log(wmdf.to_markdown(index=False))

primeras = [json.loads(l)['timestamp'] for l in open(PARTS[0])][:5]
log(f"\n**Por qué pasa:** el dataset no está ordenado por tiempo. Las primeras 5 líneas de "
    f"`{PARTS[0].name}` son {primeras} — los 60 días vienen barajados dentro de cada archivo.")
log("\nEn el primer micro-lote Spark ya ve un evento del 31/08 y el watermark salta al máximo. "
    "Un watermark de 2 días —el valor 'razonable' por reflejo— **descarta el 85% de los eventos "
    "sin lanzar ninguna excepción**. Por eso D16 fija 60 días y resuelve la idempotencia "
    "por checkpoint y anti-join, no por watermark.")
wmdf

### 8 · Volumen por clave del mart

In [ ]:
claves=collections.Counter()
for p in PARTS:
    for line in open(p):
        d=json.loads(line)
        claves[(d['org_id'], d['timestamp'][:10], d['service'])]+=1
c=sorted(claves.values())
log("\n### Grano de `org_daily_usage_by_service`\n")
log(f"- **Claves materializadas:** {len(claves):,} de 80 × 60 × 6 = 28.800 posibles")
log(f"- **Eventos por clave:** min {c[0]} · mediana {c[len(c)//2]} · p99 {c[int(.99*len(c))]} · max {c[-1]}")
log(f"- **Sin skew**: la clave más pesada tiene {c[-1]} eventos, ninguna partición domina el tiempo total.")
log("")
log("**Particionamiento (D06):**")
log("")
log("| Esquema | Particiones | Filas/partición | Tamaño/archivo |")
log("|---|---|---|---|")
log(f"| Sin partición | 1 | {n_events:,} | ~1,5 MB |")
log(f"| **Por `event_date`** | **{len(dates)}** | **~{n_events//len(dates)}** | **~28 KB** |")
log(f"| Por `event_date` × `service` | {len(dates)*6} | ~{n_events//(len(dates)*6)} | ~5 KB |")
log(f"| Por `event_date` × `service` × `region` | {len(dates)*6*7} | ~{n_events//(len(dates)*6*7)} | <1 KB |")
log("")
log("Un bloque HDFS son 128 MB (clase 04). Incluso la mejor opción produce archivos 4.000× más chicos: "
    "estamos en pleno *small files problem*. Se elige `event_date` porque las 5 consultas obligatorias "
    "filtran por fecha y ninguna por servicio solo.")

### 9 · Guardar la evidencia
El reporte se versiona en `evidence/`.

In [ ]:
out = Path('../evidence/profiling_landing.md')
out.parent.mkdir(parents=True, exist_ok=True)
header = ("# Perfilado de Landing · evidencia\n\n"
          "Salida de `notebooks/01_profiling_landing.ipynb`. "
          "Primera entrega · Cloud Provider Analytics · ITBA Big Data 2C 2026.\n\n"
          "Regenerar con: `jupyter nbconvert --execute --to notebook --inplace "
          "notebooks/01_profiling_landing.ipynb`\n\n---\n")
out.write_text(header + "\n".join(REPORT) + "\n")
print(f"Evidencia escrita en {out.resolve()} ({len(REPORT)} líneas)")